# 23CSE301 ML Capstone — EV Charging Stations

## Notebook Structure
1. **Dataset Understanding**
2. **Preprocessing**
3. **REGRESSIONS — Part 1**
4. **REGRESSIONS — Part 2**
5. **CLASSIFIER**

Preprocessing is kept separate so the prepared data can be reused by all models.

## 0. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.decomposition import PCA

RANDOM_STATE = 42
TEST_SIZE = 0.20

DATA_PATH = "EV_Charging_Stations_Feb82024.xlsx"
SHEET_NAME = "Raw"

REG_TARGET = "EV Level2 EVSE Num"
CLF_TARGET = "Access Code"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Dataset Understanding

The dataset contains EV charging-station records. The proposed regression target is **`EV Level2 EVSE Num`** and the proposed classification target is **`Access Code`** (`public` / `private`).

In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

print("Shape:", df.shape)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())


### 1.1 Structure and data types

In [ ]:
structure = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null": df.notna().sum().values,
    "Null": df.isna().sum().values,
    "Unique Values": df.nunique(dropna=True).values
})
display(structure)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))


### 1.2 Basic checks

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing Count"))

print("\nNumeric summary:")
display(df[numeric_cols].describe().T)


In [ ]:
print("Regression target:", REG_TARGET)
display(df[REG_TARGET].describe())

print("\nClassification target:", CLF_TARGET)
display(df[CLF_TARGET].value_counts(dropna=False))


## 2. Preprocessing

### Order
1. Remove exact duplicates
2. Remove rows with missing target
3. Select useful predictors and avoid leakage
4. Train/test split
5. Detect numeric outliers using **IQR**
6. Cap outliers using training-derived IQR limits
7. Fill numeric missing values using **KNN Imputer**
8. Fill categorical missing values
9. **One-Hot Encode** categorical variables
10. **Standardize** numeric/encoded features
11. Apply **PCA** for dimensionality reduction

> Preprocessing is fitted on the training data only to avoid data leakage.

### 2.1 Basic cleaning and feature engineering

In [ ]:
data = df.copy()

before = len(data)
data = data.drop_duplicates().copy()
print("Duplicates removed:", before - len(data))

if "Open Date" in data.columns:
    open_date = pd.to_datetime(data["Open Date"], errors="coerce")
    data["Open Year"] = open_date.dt.year
    data["Station Age"] = 2024 - data["Open Year"]

if "EV Connector Types" in data.columns:
    data["Num Connector Types"] = (
        data["EV Connector Types"].fillna("").astype(str).str.strip()
        .apply(lambda x: 0 if not x else len(set(x.split())))
    )

if "EV Network" in data.columns:
    data["Is Networked"] = (
        data["EV Network"].fillna("Non-Networked").astype(str).str.lower()
        .ne("non-networked").astype(int)
    )

if "EV Pricing" in data.columns:
    data["Has Pricing"] = data["EV Pricing"].notna().astype(int)

if "EV Workplace Charging" in data.columns:
    data["Has Workplace Charging"] = (
        pd.to_numeric(data["EV Workplace Charging"], errors="coerce").fillna(0) > 0
    ).astype(int)

print("Shape after cleaning:", data.shape)


### 2.2 Feature selection

In [ ]:
# Regression: remove target and direct charger-count fields to prevent leakage
reg_features = [
    c for c in [
        "Latitude", "Longitude", "Open Year", "Station Age",
        "Num Connector Types", "Is Networked", "Has Pricing",
        "Has Workplace Charging", "Access Code", "State", "Facility Type",
        "EV Network"
    ] if c in data.columns and c not in {
        REG_TARGET, "EV Level1 EVSE Num", "EV DC Fast Count"
    }]

# Classification: remove target and target-defining field
clf_features = [
    c for c in [
        "Latitude", "Longitude", "Open Year", "Station Age",
        "Num Connector Types", "Is Networked", "Has Pricing",
        "Has Workplace Charging", "State", "Facility Type",
        "EV Network", "EV Connector Types"
    ] if c in data.columns and c not in {CLF_TARGET, "Access Detail Code"}]

print("Regression features:", reg_features)
print("\nClassification features:", clf_features)


### 2.3 Train/test split

In [ ]:
reg_df = data[reg_features + [REG_TARGET]].dropna(subset=[REG_TARGET]).copy()
X_reg = reg_df[reg_features]
y_reg = pd.to_numeric(reg_df[REG_TARGET], errors="coerce")

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

clf_df = data[clf_features + [CLF_TARGET]].dropna(subset=[CLF_TARGET]).copy()
X_clf = clf_df[clf_features]
y_clf = clf_df[CLF_TARGET].astype(str)

X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=y_clf
)

print("Regression:", X_reg_train.shape, X_reg_test.shape)
print("Classification:", X_clf_train.shape, X_clf_test.shape)


### 2.4 Outlier detection — IQR

**IQR = Q3 − Q1**

Potential outlier = value `< Q1 − 1.5×IQR` or `> Q3 + 1.5×IQR`.

In [ ]:
def iqr_report(X, numeric_columns):
    rows = []
    for col in numeric_columns:
        s = pd.to_numeric(X[col], errors="coerce")
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = ((s < low) | (s > high)).sum()
        rows.append({
            "Feature": col, "Q1": q1, "Q3": q3, "IQR": iqr,
            "Lower Bound": low, "Upper Bound": high, "Outliers": int(count)
        })
    return pd.DataFrame(rows)

reg_num = X_reg_train.select_dtypes(include=np.number).columns.tolist()
clf_num = X_clf_train.select_dtypes(include=np.number).columns.tolist()

print("Regression outliers")
display(iqr_report(X_reg_train, reg_num))

print("Classification outliers")
display(iqr_report(X_clf_train, clf_num))


### 2.5 IQR outlier capping

In [ ]:
def fit_iqr_caps(X, numeric_columns):
    caps = {}
    for col in numeric_columns:
        s = pd.to_numeric(X[col], errors="coerce")
        q1, q3 = s.quantile([0.25, 0.75])
        iqr = q3 - q1
        caps[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return caps

def apply_iqr_caps(X, caps):
    X = X.copy()
    for col, (low, high) in caps.items():
        X[col] = pd.to_numeric(X[col], errors="coerce").clip(low, high)
    return X

reg_caps = fit_iqr_caps(X_reg_train, reg_num)
clf_caps = fit_iqr_caps(X_clf_train, clf_num)

X_reg_train = apply_iqr_caps(X_reg_train, reg_caps)
X_reg_test = apply_iqr_caps(X_reg_test, reg_caps)
X_clf_train = apply_iqr_caps(X_clf_train, clf_caps)
X_clf_test = apply_iqr_caps(X_clf_test, clf_caps)

print("IQR capping completed.")


### 2.6 KNN Imputation + One-Hot Encoding + Standardization

In [ ]:
reg_num = X_reg_train.select_dtypes(include=np.number).columns.tolist()
reg_cat = X_reg_train.select_dtypes(exclude=np.number).columns.tolist()

clf_num = X_clf_train.select_dtypes(include=np.number).columns.tolist()
clf_cat = X_clf_train.select_dtypes(exclude=np.number).columns.tolist()

def make_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", Pipeline([
            ("knn_imputer", KNNImputer(n_neighbors=5)),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols)
    ])

reg_preprocessor = make_preprocessor(reg_num, reg_cat)
clf_preprocessor = make_preprocessor(clf_num, clf_cat)

print("Regression numeric:", reg_num)
print("Regression categorical:", reg_cat)
print("\nClassification numeric:", clf_num)
print("Classification categorical:", clf_cat)


### 2.7 Transform data

In [ ]:
X_reg_train_p = reg_preprocessor.fit_transform(X_reg_train)
X_reg_test_p = reg_preprocessor.transform(X_reg_test)

X_clf_train_p = clf_preprocessor.fit_transform(X_clf_train)
X_clf_test_p = clf_preprocessor.transform(X_clf_test)

print("Regression processed shape:", X_reg_train_p.shape)
print("Classification processed shape:", X_clf_train_p.shape)


### 2.8 PCA

In [ ]:
# Retain 95% of the variance
pca_reg = PCA(n_components=0.95, svd_solver="full")
pca_clf = PCA(n_components=0.95, svd_solver="full")

X_reg_train_pca = pca_reg.fit_transform(X_reg_train_p)
X_reg_test_pca = pca_reg.transform(X_reg_test_p)

X_clf_train_pca = pca_clf.fit_transform(X_clf_train_p)
X_clf_test_pca = pca_clf.transform(X_clf_test_p)

print("Regression PCA shape:", X_reg_train_pca.shape)
print("Classification PCA shape:", X_clf_train_pca.shape)
print("Regression variance retained:", round(pca_reg.explained_variance_ratio_.sum(), 4))
print("Classification variance retained:", round(pca_clf.explained_variance_ratio_.sum(), 4))


## 3. REGRESSIONS — Part 1

### Models to fill later
1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. ElasticNet
5. Polynomial Regression

**Metrics:** R², RMSE, MAE

Use:
`X_reg_train_pca`, `X_reg_test_pca`, `y_reg_train`, `y_reg_test`

In [ ]:
# REGRESSIONS — PART 1
# 1. Linear Regression
# 2. Ridge Regression
# 3. Lasso Regression
# 4. ElasticNet
# 5. Polynomial Regression

# Add model -> fit -> predict -> evaluate code here.


## 4. REGRESSIONS — Part 2

### Models to fill later
6. Decision Tree Regressor
7. Random Forest Regressor
8. Gradient Boosting Regressor
9. Support Vector Regressor (SVR)
10. K-Nearest Neighbors Regressor

**Metrics:** R², RMSE, MAE

Use:
`X_reg_train_pca`, `X_reg_test_pca`, `y_reg_train`, `y_reg_test`

In [ ]:
# REGRESSIONS — PART 2
# 6. Decision Tree Regressor
# 7. Random Forest Regressor
# 8. Gradient Boosting Regressor
# 9. Support Vector Regressor
# 10. K-Nearest Neighbors Regressor

# Add model -> fit -> predict -> evaluate code here.


## 5. CLASSIFIER

### Models to fill later
1. Logistic Regression
2. K-Nearest Neighbors
3. Gaussian Naive Bayes
4. Decision Tree Classifier
5. Support Vector Classifier (SVC)

**Metrics:** Accuracy, Precision, Recall, F1-score, Confusion Matrix, ROC-AUC

Use:
`X_clf_train_pca`, `X_clf_test_pca`, `y_clf_train`, `y_clf_test`

In [ ]:
# CLASSIFIER
# 1. Logistic Regression
# 2. K-Nearest Neighbors
# 3. Gaussian Naive Bayes
# 4. Decision Tree Classifier
# 5. Support Vector Classifier

# Add model -> fit -> predict -> evaluate code here.


## Preprocessing Checklist

- Dataset structure and data types
- Missing-value inspection
- Duplicate removal
- Target definition
- Target leakage control
- Train/test split
- IQR outlier detection
- IQR outlier capping
- KNN imputation for numeric values
- Categorical imputation
- One-hot encoding
- Standardization
- PCA
- Regression shells
- Classifier shell
